# Robot session

Everything needed to bring the robot up, calibrate it, and hand coordinates to a
picking routine. Run the cells in order the first time; afterwards the
calibration sections can be re-run on their own.

The order in section 1 is not optional. `move_to_coordinates`, `move_relative`
and `get_position` all require both a run and a loaded pipette, so anything that
moves the robot fails until `create_run` and `load_pipette` have happened.

## 0. Setup

In [1]:
# Shared setup for both notebooks: sets MICROPICK_ROOT and imports the common
# surface. See notebooks/bench_setup.py.
from bench_setup import *

paths.ensure_layout()
print(paths.describe())
print("\nprofiles:", store.list_profiles())

  root       ok       C:\Users\ivand\Desktop\micropick
  profiles   ok       C:\Users\ivand\Desktop\micropick\profiles
  labware    ok       C:\Users\ivand\Desktop\micropick\labware
  ml_models  ok       C:\Users\ivand\Desktop\micropick\ml_models
  outputs    ok       C:\Users\ivand\Desktop\micropick\outputs
  logs       ok       C:\Users\ivand\Desktop\micropick\logs

profiles: ['lab_main']


In [2]:
from micropick.hardware import labware
for d in labware.list_definitions().values():
    print(d)

greiner_1536_wellplate_12.6ul v1 (custom_beta, local), 1536 wells
wide_bore_200ul v1 (custom_beta, local), 96 wells
vwr_96_tiprack_200ul_xl v1 (custom_beta, local), 96 wells


### Profile

One profile per installation. Created once, then loaded on every later run.

In [ ]:
# PROFILE = "lab_main"

# profile = store.create_profile(PROFILE, camera_label="overview_cam",
#                                notes="OT-2, gantry camera", exist_ok=True)
# print(profile)
# print("positions:", sorted(profile.positions) or "none yet")

In [2]:
PROFILE = "lab_main"
profile = store.load_profile(PROFILE)
print(profile)
print("positions:", sorted(profile.positions) or "none yet")

<Profile 'lab_main' (degree 3, 49 poses) at C:\Users\ivand\Desktop\micropick\profiles\lab_main>
positions: ['observe', 'tip_calib']


### Cameras, first run only

Names must match what the operating system reports. Focus and exposure are
re-applied every time a camera is opened, which is what keeps a pixel map valid
across restarts: the map is only correct for the focus it was fitted at.

In [ ]:
from micropick.hardware import devices
print(*devices.list_devices(), sep="\n")

In [ ]:
profile.cameras = {
    "overview_cam": CameraSpec(
        device_name="20MP U3 Camera",
        resolutions=[[1280,720],
                    [1920,1080],
                    [2048,1536],
                    [2592,1944],
                    [3840,2160],
                    [4000,3000],
                    [4608,3456],
                    [5120,3840]],
        default_resolution=[2592, 1944], fps=30, fourcc="MJPG",
        controls={"auto_exposure": "manual"},
        notes="on the gantry, manual focus ring"),

    "underview_cam": CameraSpec(
        device_name="Arducam B0478 (USB3 48MP)",
        resolutions=[[1280,720],
                    [1920,1080],
                    [2000,1500],
                    [3840,2160],
                    [4000,3000],
                    [8000,6000]],
        default_resolution=[4000, 3000], fps=30, fourcc="MJPG",
        controls={"autofocus": 0, "focus": 920, "auto_exposure": "manual"},
        notes="tip calibration module, motorised focus"),
}
profile.save_cameras()
print(*profile.cameras.values(), sep="\n")

## 1. Robot

In [3]:
openapi = ot2_api.OpentronsAPI()
openapi.add_slot_offsets([5, 8, 9], (0, 0, 64.2))

In [4]:
openapi.toggle_lights()

<Response [200]>

In [4]:
# Use to restore labware and general run information after the notebook crashes
r = openapi.get_run_info()

Total number of runs: 20
Current run ID: d92624b9-4d07-486c-9b9a-181b67dc0fca
Current run status: idle


In [7]:
# Once after power on.
openapi.home_robot()

Request status:
<Response [200]>
{
  "message": "Homing robot."
}


<Response [200]>

In [8]:
openapi.create_run()
openapi.load_pipette()
print("run:", openapi.run_id, " pipette:", openapi.pipette_id)

Request status:
<Response [201]>
{
  "data": {
    "id": "d92624b9-4d07-486c-9b9a-181b67dc0fca",
    "ok": true,
    "createdAt": "2025-07-04T20:41:44.680891Z",
    "status": "idle",
    "current": true,
    "actions": [],
    "errors": [],
    "hasEverEnteredErrorRecovery": false,
    "pipettes": [],
    "modules": [],
    "labware": [],
    "liquids": [],
    "liquidClasses": [],
    "labwareOffsets": [],
    "runTimeParameters": [],
    "outputFileIds": []
  }
}
Request status:
<Response [201]>
{
  "data": {
    "id": "393f0551-35b9-42b6-8f79-9d8718c2dfcb",
    "createdAt": "2025-07-04T20:41:45.376776Z",
    "commandType": "loadPipette",
    "key": "393f0551-35b9-42b6-8f79-9d8718c2dfcb",
    "status": "succeeded",
    "params": {
      "pipetteName": "p300_single_gen2",
      "mount": "left"
    },
    "result": {
      "pipetteId": "7b048808-c034-4e91-902c-3c2b6cafd547"
    },
    "startedAt": "2025-07-04T20:41:45.380006Z",
    "completedAt": "2025-07-04T20:41:47.357745Z",
    "int

### Labware

Definitions live in `labware/` and are uploaded into the current run. This has
to happen again after every `create_run`. Names and namespaces come from the
files themselves.

In [9]:
for d in labware.list_definitions().values():
    print(d)

greiner_1536_wellplate_12.6ul v1 (custom_beta, local), 1536 wells
wide_bore_200ul v1 (custom_beta, local), 96 wells
vwr_96_tiprack_200ul_xl v1 (custom_beta, local), 96 wells


In [10]:
labware.ensure_definitions(openapi)

TIP_RACK = "vwr_96_tiprack_200ul_xl"
labware.load_labware(openapi, TIP_RACK, 10)

uploaded greiner_1536_wellplate_12.6ul v1 (custom_beta, local), 1536 wells
uploaded wide_bore_200ul v1 (custom_beta, local), 96 wells
uploaded vwr_96_tiprack_200ul_xl v1 (custom_beta, local), 96 wells


<Response [201]>

In [11]:
openapi.pick_up_tip(openapi.labware_dct["10"], "A4")

<Response [201]>

In [45]:
WELL_PLATE = "corning_96_wellplate_360ul_flat"
# WELL_PLATE = "corning_6_wellplate_16.8ml_flat"
# WELL_PLATE = "corning_24_wellplate_3.4ml_flat"
# WELL_PLATE = "corning_384_wellplate_112ul_flat"
r = openapi.load_labware(WELL_PLATE, 5, namespace='opentrons',verbose=True)

Offset (0, 0, 64.2) added to run for corning_96_wellplate_360ul_flat in slot 5.
Labware URI:
opentrons/corning_96_wellplate_360ul_flat/1

Check offset before using ...
Labware ID:
41a6d8e4-ed5a-4502-b4a4-480d34480662



In [44]:
openapi.move_labware(openapi.labware_dct['5'], 'offDeck')

<Response [201]>

## 2. Cameras

In [5]:
cams = CameraManager.from_profile(profile)
over_cam = cams.open("overview_cam")
print(over_cam)

overview_cam: [2] 20MP U3 Camera  2592x1944
  NOT applied: auto_exposure: asked 0.25, got 0
<BackgroundCamera 'overview_cam' 2592x1944 running, 0 frames>


In [ ]:
# The lower camera is only needed for tip calibration; open it there.
# cams.close("underview_cam")

In [6]:
def preview(camera, window="preview", size=(1348, 1011)):
    """Живой просмотр. Esc или q закрывает, s сохраняет кадр в outputs/images."""
    cv2.namedWindow(window, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window, *size)
    try:
        while True:
            ok, frame = camera.read()
            if not ok:
                continue
            vis = frame.copy()
            h, w = vis.shape[:2]
            cv2.drawMarker(vis, (w // 2, h // 2), (0, 0, 255), cv2.MARKER_CROSS, 60, 2)
            cv2.putText(vis, f"{w}x{h}  {camera.measure_fps(0.0) if False else ''}"
                             f"frames {camera.frame_count}", (20, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3)
            cv2.imshow(window, vis)
            key = cv2.waitKey(20) & 0xFF
            if key in (27, ord("q")):
                break
            if key == ord("s"):
                path = paths.images_dir() / f"{time.strftime('%H%M%S')}.png"
                cv2.imwrite(str(path), frame)
                print("saved", path)
    finally:
        cv2.destroyWindow(window)

# preview(over_cam)

### Jogging

Arrows or WASD move x and y, `q` and `e` move z, `+` and `-` change the step,
space saves a position, `u` undoes the last step, Enter finishes. The window
must have focus, so a stray keystroke in the notebook cannot drive the robot.

Limits are soft. Outside them the robot can always move back toward the working
area, only further out is refused.

In [7]:
LIMITS = Limits(x=(0, 380), y=(0, 350), z=(0.1, 150))

def jog(title="", camera=None, step=1.0):
    ctrl = JogController(openapi, limits=LIMITS, step=step)
    pos = jog_in_window(ctrl, camera or over_cam, title=title)
    print("stopped at", tuple(round(v, 2) for v in pos))
    return pos

In [16]:
jog()

stopped at (42.0, 344.5, 117.6)


(42.004465697898276, 344.50246866029056, 117.60000000000001)

## 3. Camera calibration

Fits lens distortion and the camera-to-robot relationship together from one
sweep of a static ArUco marker. No chessboard and no undistortion stage.

Redo it after any change to focus, zoom, camera height, or the height of the
plane the objects sit on.

In [20]:
openapi.toggle_lights()

<Response [200]>

In [17]:
MARKER_SIDE_MM = 6.8

aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250)
params = cv2.aruco.DetectorParameters()
params.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_SUBPIX
detector = cv2.aruco.ArucoDetector(aruco_dict, params)

Put the marker roughly in the centre of the frame and set Z to the height you
actually image the dish at. The sweep keeps whatever Z it starts from.

In [23]:
jog("centre the marker, set the working Z, then Enter")

stopped at (182.0, 161.5, 117.6)


(182.00446569789827, 161.50246866029056, 117.60000000000001)

In [24]:
pmap, report, sweep = calibrate_camera(
    openapi, over_cam, detector,
    marker_side_mm=MARKER_SIDE_MM, grid_n=7, degree=3,
    on_progress=lambda i, n: print(f"  {i}/{n}", end="\r"))

print("\n")
print(report)

measuring scale
  probe 1: 38 px, adjusting step to 13.61 mm
  scale: 13.606 mm moved the marker 518.7 px = 26.23 um/px
  field of view 68.0 x 51.0 mm, marker 259 px
planning: 7x7 poses, +/-30.0 x +/-21.0 mm, expected coverage 98 % x 96 % of the frame
sweeping 49 poses
  collected 49/49 poses, tracking id 1


degree 3, 49 poses, 196 points
  residual   mean    22.5  max    62.1 um
  held out   mean    24.0  max    83.3 um
  coverage   u [48, 2542]  v [28, 1871]  (96 % x 95 % of frame)
  scale      centre 26.00  edge 28.67 um/px (+10.3 %)
  track side 6.7473 mm


`track side` should come out near the printed marker size. The fit never uses
it, so agreement is independent evidence the sweep was good.

Degrees 1 and 2 should give identical numbers: radial distortion is cubic in
image coordinates, so a quadratic reduces exactly to an affine fit. A difference
between them would mean something other than lens distortion is in the data.

In [25]:
print(compare_degrees(sweep.track_px, sweep.gantry, sweep.image_size,
                      marker_side_mm=MARKER_SIDE_MM))

degree |    residual, um    |    held out, um    | track side
  1    |   219.9 /   985.7  |   229.8 /  1040.2 | 6.6857 mm
  2    |   218.5 /   874.2  |   243.4 /  1107.7 | 6.6857 mm
  3    |    22.5 /    62.1  |    24.0 /    83.3 | 6.7473 mm
  4    |    21.9 /    53.7  |    24.5 /   105.7 | 6.7473 mm
nominal track side 6.8000 mm


In [26]:
profile.calibration.pixel_map = pmap.to_config()
profile.save_calibration()
sweep.save(str(paths.fixtures_dir() / f"sweep_{time.strftime('%Y%m%d_%H%M')}.npz"))
print("saved")

saved


### Closed-loop check

Ask the map where the marker is, drive there, and see how far it lands from the
reference pixel. This is the only test that includes the robot.

Read the spread, not the absolute value. A consistent offset in one direction
with a small spread is the camera-to-tip constant and belongs to the pipette
offset; scatter is the map and the robot's repeatability.

In [27]:
def marker_centre_now():
    frame = over_cam.read_after(time.monotonic())
    corners, ids, _ = detector.detectMarkers(frame)
    if ids is None or len(corners) == 0:
        return None
    return corners[0].reshape(4, 2).mean(axis=0)

origin = xyz(openapi)
errors = []
for dx, dy in [(0, 0), (10, 7), (-12, -8), (18, -11), (-20, 12)]:
    openapi.move_to_coordinates((origin[0] + dx, origin[1] + dy, origin[2]-1),
                                min_z_height=1, verbose=False)
    time.sleep(0.4)
    g = xyz(openapi)[:2]                       # read next to the frame
    q = marker_centre_now()
    if q is None or not pmap.covers(*q):
        print(f"({dx:+3.0f},{dy:+3.0f}) not usable")
        continue

    target = pmap.to_robot(q[0], q[1], g)
    openapi.move_to_coordinates((target[0], target[1], origin[2]-1),
                                min_z_height=1, verbose=False)
    time.sleep(0.4)
    q2 = marker_centre_now()
    if q2 is None:
        continue
    err_px = q2 - np.array(pmap.config.ref)
    scale = float(np.mean(pmap.mm_per_px(*q2)))
    errors.append(err_px * scale)
    print(f"({dx:+3.0f},{dy:+3.0f})  residual {err_px[0]:+7.1f}, {err_px[1]:+7.1f} px"
          f"  = {np.linalg.norm(err_px) * scale * 1000:6.0f} um")

if errors:
    e = np.array(errors)
    print(f"\nbias   {e.mean(0)[0]*1000:+.0f}, {e.mean(0)[1]*1000:+.0f} um"
          f"   (constant, belongs to the pipette offset)")
    print(f"spread {np.linalg.norm(e - e.mean(0), axis=1).max()*1000:.0f} um max"
          f"   (this is the map plus robot repeatability)")

( +0, +0)  residual    -0.1,    -0.7 px  =     19 um
(+10, +7)  residual    -0.9,    +0.5 px  =     26 um
(-12, -8)  residual    +1.3,    -1.5 px  =     51 um
(+18,-11)  residual    -0.9,    -1.4 px  =     43 um
(-20,+12)  residual    +1.5,    +0.5 px  =     40 um

bias   +4, -14 um   (constant, belongs to the pipette offset)
spread 43 um max   (this is the map plus robot repeatability)


## 4. Pipette offset calibration

Redo this whenever a tip is picked up: every tip seats differently.

The upper camera locates the crosshair disc, the robot drives there using the
current offset, and the lower camera measures how far the tip actually is. The
gantry is parked a few millimetres to one side first, otherwise the tip covers
the crosshair and neither can be measured.

On a new installation the offset must be filled in roughly by hand first,
measured with a ruler. The routine drives to where it thinks the target is
before looking, so an offset that is wrong by tens of millimetres puts the tip
outside the lower camera's view and the run cannot recover.

In [28]:
from ultralytics import YOLO

tip_model = YOLO(str(paths.ml_models_dir() / profile.calibration.tip_target.model_file))
tip_detector = TipDetector(tip_model,
                           imgsz=profile.calibration.tip_target.imgsz,
                           conf=profile.calibration.tip_target.conf)
under_cam = cams.open("underview_cam")
print(under_cam)

underview_cam: [0] Arducam B0478 (USB3 48MP)  4000x3000
  NOT applied: auto_exposure: asked 0.25, got 0, autofocus: asked 0, got 1, focus: asked 920, got 1
<BackgroundCamera 'underview_cam' 4000x3000 running, 0 frames>


In [29]:
openapi.toggle_lights()

<Response [200]>

In [38]:
preview(under_cam)

First run only: fill in a rough offset measured with a ruler, and teach the
position of the calibration module.

In [ ]:
if profile.calibration.pipette_offset is None:
    profile.calibration.pipette_offset = PipetteOffset(
        dx=16.0, dy=60.0, tip_type="vwr_200ul_xl", method="manual")
    profile.save_calibration(backup=False)
print(profile.calibration.pipette_offset)

In [31]:
# Teach where the crosshair disc is, once. Skip if it is already stored.
if "tip_calib" not in profile.positions:
    jog("bring the crosshair disc under the camera, then Enter")
    profile.remember("tip_calib", xyz(openapi))
print("tip_calib:", profile.where("tip_calib"))

tip_calib: (281.87353703094635, 161.75594045486756, 114.50000000000001)


In [32]:
target = profile.calibration.tip_target
openapi.move_to_coordinates(profile.where("tip_calib"),
                            min_z_height=target.module_height - 0.1, verbose=False)
time.sleep(0.5)

`manual_touch_up` runs after the automatic correction, with the lower camera
live. Nudge the tip onto the crosshair with a small step if the result is not
good enough, then press Enter. Whatever it moves is included, because the offset
is read from the final pose rather than from the commanded moves.

In [33]:
def touch_up(robot, camera, view):
    ctrl = JogController(robot, limits=LIMITS, step=0.05)
    jog_in_window(ctrl, camera, window="tip",
                  title="nudge the tip onto the crosshair, then Enter")

current = profile.calibration.pipette_offset
result = calibrate_pipette_offset(
    openapi, over_cam, under_cam, tip_detector, pmap,
    target=target,
    current_offset=(current.dx, current.dy),
    frames=7,
    tip_type=current.tip_type,
    manual_touch_up=touch_up)          # pass None to skip the manual step

print()
print(result)

upper camera: locating the crosshair
  crosshair at [     281.95      161.91] mm, spread 0.1 px over 7 frames
lower camera: measuring the tip
  residual [       30.8      -111.7] px = [     -2.671       0.737] mm, spread 0.6 px
  verification unavailable: no usable reading in 7 frames. Last problem: found 4 crosshairs, need the centre plus four neighbours; check lighting and focus
  manual touch-up moved [      -0.05       -0.05] mm
  homography: homography from 5 points: reprojection mean 0.46 px, max 0.99 px

offset          : dx  +16.817  dy  +60.999 mm
correction      :   -2.671,   +0.737 mm
upper camera    : 7 frames, spread 0.1 px
lower camera    : 7 frames, spread 0.6 px, 23.90 um/px
change from last: 0.741 mm
manual touch-up : -0.050, -0.050 mm


In [34]:
profile.calibration.pipette_offset = result.offset
profile.save_calibration(backup=False)
print("saved:", profile.calibration.pipette_offset)

saved: dx=16.816866066914827 dy=60.99943258534 tip_type='vwr_200ul_xl' measured_at=datetime.datetime(2026, 8, 11, 0, 29, 4, 376727, tzinfo=datetime.timezone.utc) method='auto+manual' residual_mm=None n_samples=7 spread_mm=0.01524732945319168


### Camera homography

Maps upper-camera pixels to the lower camera, for placing a ROI box on the
pickup video. It comes free from the pipette calibration (both cameras saw the
same disc), and also runs on its own with no pipette and no moves. **It is valid
only for the marker plane and only at the gantry pose the upper frame was taken
at** — cuboids sit lower on the dish, so it is a coarse viewing aid, never a
positioning tool. Optional: skip it if there is no lower camera.

In [35]:
# The pipette calibration above already saw the disc in both cameras, so it
# produced a homography for free. Keep it, or use the standalone cells below.
if result.homography is not None:
    profile.calibration.homography = result.homography
    profile.save_calibration(backup=False)
    print(result.homography_report)
else:
    print("no usable homography from the pipette run; try the standalone cell")

homography from 5 points: reprojection mean 0.46 px, max 0.99 px


In [ ]:
# Doesn't work, needs robot movement

from micropick.workflows.calibrate_homography import calibrate_homography

openapi.move_to_coordinates(profile.where("tip_calib"),
                            min_z_height=target.module_height - 0.1, verbose=False)
time.sleep(0.5)
# Standalone: the disc under both cameras, no pipette, no moves. Reads the
# gantry pose the upper frame was taken at and stores it with the matrix.
hres = calibrate_homography(openapi, over_cam, under_cam, tip_detector,
                            target=profile.calibration.tip_target)
print(hres.report)

upper camera: locating the crosshair
lower camera: locating the crosshair


TipCalibrationError: no usable reading in 7 frames. Last problem: found 4 crosshairs, need the centre plus four neighbours; check lighting and focus

In [ ]:
# Per-point reprojection error, in lower-camera pixels. A large max means a
# crosshair was mis-detected or the correspondence is wrong.
for i, e in enumerate(hres.report.per_point_px):
    print(f"  point {i}: {e:.2f} px")
print(f"mean {hres.report.reproj_mean_px:.2f} px, "
      f"max {hres.report.reproj_max_px:.2f} px")

In [39]:
profile.calibration.homography = hres.homography
profile.save_calibration(backup=False)
print("saved homography, captured at gantry", hres.homography.gantry_xy)

NameError: name 'hres' is not defined

## 5. Using the calibration

`pixel_to_robot` is the one function the picking code needs. The gantry pose has
to be read next to the frame the pixel came from: the pose is part of the
conversion, not a correction applied afterwards.

`mm_per_px` replaces the old global size ratio. Scale varies by several percent
across the frame, so a single number misreports objects near the edges.

In [40]:
profile = store.load_profile(PROFILE)
profile.require_calibration()
pmap = PixelMap.from_config(profile.pixel_map)
off = profile.calibration.pipette_offset
tip_offset = np.array([off.dx, off.dy])

problems = profile.pixel_map.check_camera(over_cam.resolution)
if problems:
    raise RuntimeError("the calibration does not match the camera: " + "; ".join(problems))

def pixel_to_robot(u, v, gantry_xy):
    """Robot coordinates that put the pipette tip on the pixel (u, v)."""
    if not pmap.covers(u, v):
        raise ValueError(f"pixel ({u:.0f}, {v:.0f}) is outside the calibrated area")
    return pmap.to_robot(u, v, gantry_xy) + tip_offset

def area_mm2(area_px, u, v):
    su, sv = pmap.mm_per_px(u, v)
    return area_px * su * sv

print("ready:", pmap.config.degree, "degree map,",
      f"holdout {pmap.config.holdout_mean_um:.1f} um,",
      f"offset ({off.dx:.2f}, {off.dy:.2f}) mm")

ready: 3 degree map, holdout 24.0 um, offset (16.82, 61.00) mm


In [ ]:
# Example: convert one detection.
g = xyz(openapi)[:2]
frame = over_cam.read_after(time.monotonic())
# u, v = ...detect something...
# tx, ty = pixel_to_robot(u, v, g)

In [42]:
def click_to_go(z=None, snap_px=60, conf=0.25, imgsz=2016, move=True):
    """
    Клик по кресту -> пипетка едет туда.

    Клик привязывается к ближайшей детекции, а не к сырым координатам курсора:
    попасть мышью в пиксель невозможно, а центр бокса модели субпиксельный.

    После переезда тот же крест находится заново, и его координаты считаются
    из новой позы гантри. Совпадение с прежними это и есть проверка карты по
    полю, для неё не нужно ничего измерять руками.

    Клавиши: d пересчитать детекции, r сбросить статистику, Esc выход.
    """
    win = "click to go"
    state = {"dets": [], "click": None, "frame": None, "gantry": None}
    history = []

    def on_mouse(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            state["click"] = (x, y)

        if event == cv2.EVENT_RBUTTONDOWN:
            openapi.move_to_coordinates(profile.where("tip_calib"))

    def detect_now():
        g = np.array(xyz(openapi)[:2])
        frame = over_cam.read_after(time.monotonic())
        res = tip_model.predict(source=frame[..., ::-1], conf=conf, imgsz=imgsz,
                                save=False, verbose=False)
        pts = []
        for r in res:
            for b in r.boxes:
                if tip_model.names[int(b.cls[0])] != "point":
                    continue
                x1, y1, x2, y2 = (float(v) for v in b.xyxy[0])
                p = np.array([(x1 + x2) / 2, (y1 + y2) / 2])
                if pmap.covers(*p):
                    pts.append((p, pmap.to_robot(p[0], p[1], g)))
        state["dets"], state["gantry"] = pts, g
        return pts

    print("детекция...")
    detect_now()
    print(f"найдено {len(state['dets'])} крестов")

    cv2.namedWindow(win, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win, 1348, 1011)
    cv2.setMouseCallback(win, on_mouse)
    try:
        while True:
            ok, frame = over_cam.read()
            if not ok:
                continue
            vis = frame.copy()
            h, w = vis.shape[:2]
            cv2.drawMarker(vis, tuple(np.int32(pmap.config.ref)), (0, 0, 255),
                           cv2.MARKER_CROSS, 60, 2)
            for p, world in state["dets"]:
                cv2.circle(vis, tuple(np.int32(p)), 14, (0, 255, 0), 2)
            cv2.putText(vis, f"{len(state['dets'])} crosses   click one   "
                             f"d=redetect  r=reset  Esc=quit", (20, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
            if history:
                e = np.array(history)
                cv2.putText(vis, f"map consistency: mean {e.mean()*1000:.0f} um, "
                                 f"max {e.max()*1000:.0f} um  (n={len(e)})",
                            (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 1.2,
                            (0, 200, 255), 2)
            cv2.imshow(win, vis)

            key = cv2.waitKey(20) & 0xFF
            if key == 27:
                break
            if key == ord("d"):
                detect_now(); print(f"найдено {len(state['dets'])}")
            if key == ord("r"):
                history.clear()

            if state["click"] is None:
                continue
            cx, cy = state["click"]
            state["click"] = None
            if not state["dets"]:
                print("нет детекций, нажми d"); continue

            # привязка к ближайшему кресту в координатах отображаемого кадра
            # scale_x = frame.shape[1] / cv2.getWindowImageRect(win)[2]
            # scale_y = frame.shape[0] / cv2.getWindowImageRect(win)[3]
            # click_px = np.array([cx * scale_x, cy * scale_y])
            click_px = np.array([cx, cy])
            d = [np.linalg.norm(p - click_px) for p, _ in state["dets"]]
            k = int(np.argmin(d))
            # if d[k] > snap_px * max(scale_x, scale_y):
            #     print(f"мимо креста ({d[k]:.0f} px до ближайшего)"); continue
            if d[k] > snap_px:
                print(f"мимо креста ({d[k]:.0f} px до ближайшего)"); continue

            px, world = state["dets"][k]
            target = world + tip_offset
            print(f"\nкрест на пикселе ({px[0]:.0f}, {px[1]:.0f})")
            print(f"  координата креста : {world.round(3)}")
            print(f"  цель для пипетки  : {target.round(3)}")
            if not move:
                continue

            goto_xy(openapi, target[0], target[1])
            if z is not None:
                openapi.move_to_coordinates((target[0], target[1], z), min_z_height=1, verbose=False)
                # move_to(openapi, (target[0], target[1], z))

            # тот же крест заново, из новой позы
            after = detect_now()
            if after:
                worlds = np.array([wr for _, wr in after])
                j = int(np.argmin(np.linalg.norm(worlds - world, axis=1)))
                drift = float(np.linalg.norm(worlds[j] - world))
                history.append(drift)
                print(f"  тот же крест из новой позы: {worlds[j].round(3)}")
                print(f"  расхождение карты: {drift*1000:.0f} um")
    finally:
        cv2.destroyWindow(win)

    if history:
        e = np.array(history)
        print(f"\nсогласованность карты по {len(e)} переездам: "
              f"среднее {e.mean()*1000:.0f} мкм, максимум {e.max()*1000:.0f} мкм")
    return history



In [ ]:
jog("Alignment")

In [43]:
drift = click_to_go(z = 67.0)

детекция...
найдено 5 крестов

крест на пикселе (2069, 970)
  координата креста : [     302.15      161.87]
  цель для пипетки  : [     318.97      222.87]
Request status:
<Response [201]>
{
  "data": {
    "id": "5822a7ea-e130-42eb-bc63-2386f4acc13e",
    "createdAt": "2025-07-04T21:02:31.517544Z",
    "commandType": "moveToCoordinates",
    "key": "5822a7ea-e130-42eb-bc63-2386f4acc13e",
    "status": "succeeded",
    "params": {
      "minimumZHeight": 20.0,
      "forceDirect": false,
      "pipetteId": "7b048808-c034-4e91-902c-3c2b6cafd547",
      "coordinates": {
        "x": 281.87353703094635,
        "y": 161.75594045486756,
        "z": 114.50000000000001
      }
    },
    "result": {
      "position": {
        "x": 281.87353703094635,
        "y": 161.75594045486756,
        "z": 114.50000000000001
      }
    },
    "startedAt": "2025-07-04T21:02:31.520306Z",
    "completedAt": "2025-07-04T21:02:32.489832Z",
    "intent": "setup",
    "notes": []
  }
}
найдено 5

крест на 

## 6. Picking cuboids

The picking session (`workflows/picking.py`) is headless: it has no loop, no
window and no keyboard. This section is the external layer that provides them.
`session.step()` performs one transition and returns an event; the run cell
holds the `while`, the pause, the stop and the display, and reads keys from the
focused window rather than global hooks, so a stray keypress in the notebook
cannot drive the robot.

Run the cells in order: detector, config, plate map, routine, (optional) logger,
then the run cell. Section 5 must have run first, so `pmap`, `over_cam` and
`profile` exist.

In [8]:
# --- Detector -----------------------------------------------------------
# The cuboid YOLO weights live in ml_models/ and are not tracked in the repo.
import threading
from ultralytics import YOLO

from micropick.core import routine as rt
from micropick.workflows.picking import PickingSession, RobotState
from micropick.viz import overlays

CUBOID_WEIGHTS = "cuboid_bbox_v4-11_best.pt"      # change to your weights file
_weights = paths.ml_models_dir() / CUBOID_WEIGHTS
if not _weights.exists():
    raise FileNotFoundError(
        f"cuboid detector weights not found: {_weights}\n"
        f"put the .pt file in {paths.ml_models_dir()} "
        f"(weights are not tracked in the repository)")
cuboid_model = YOLO(str(_weights))
print("loaded", CUBOID_WEIGHTS)

loaded cuboid_bbox_v4-11_best.pt


In [9]:
# --- Config -------------------------------------------------------------
# Picking parameters come from the profile. Edit in place for this run; the
# commented save writes the change back to picking.json.
cfg = profile.picking
cfg.miss_policy = "keep_successful"        # or "return_all"
cfg.vol = 10.0
cfg.max_batch = 10
# profile.save_picking()
print(cfg.miss_policy, "| vol", cfg.vol, "| batch", cfg.max_batch,
      "| slot", cfg.destination_slot)

keep_successful | vol 10.0 | batch 10 | slot 5


In [ ]:
# --- Plate map ----------------------------------------------------------
# The destination is described by its labware definition, not a size: the well
# names and the fill order come from the definition, so the plan cannot
# disagree with what is loaded. PLATE is a load_name in labware/ (or a stock
# Opentrons name, which needs the 'stock' extra).
# PLATE = ""

if str(cfg.destination_slot) not in openapi.labware_dct:
    labware.load_labware(openapi, WELL_PLATE, cfg.destination_slot)
labware_id = openapi.labware_dct[str(cfg.destination_slot)]

dest = rt.Destination.from_labware(WELL_PLATE, cfg.destination_slot)

# Fill the grid by hand, the old way: well_df.loc['C', 3] = 1
well_df = rt.empty_plate_table(dest)
well_df.loc['C', 3] = 1
# well_df.loc['D', 5] = 2
# ...
plan = rt.plan_from_table(well_df, dest)

NameError: name 'WELL_PLATE' is not defined

In [50]:
well_df

,1,2,3,4,5,6,7,8,9,10,11,12
A,0,0,0,0,0,0,0,0,0,0,0,0
B,0,0,0,0,0,0,0,0,0,0,0,0
C,0,0,1,0,0,0,0,0,0,0,0,0
D,0,0,0,0,0,0,0,0,0,0,0,0
E,0,0,0,0,0,0,0,0,0,0,0,0
F,0,0,0,0,0,0,0,0,0,0,0,0
G,0,0,0,0,0,0,0,0,0,0,0,0
H,0,0,0,0,0,0,0,0,0,0,0,0


In [10]:
PROGRESS = paths.outputs_dir() / f"picking_{PROFILE}.json"
# print(len(plan), "wells,", sum(plan.values()), "cuboids ->", dest)
print("progress file:", PROGRESS)

progress file: C:\Users\ivand\Desktop\micropick\outputs\picking_lab_main.json


### Routine — new run (erases saved progress)

A routine tracks how many cuboids have gone into each well and writes that to
disk after every pickup. Creating a new one starts that record from zero, so
the constructor is commented out on purpose — run it only when starting over,
not after a restart mid-plate.

In [51]:
# DANGER: uncomment to start a NEW run. This overwrites PROGRESS on disk and
# discards the record of everything already filled. Leave it commented unless
# you really are starting the plate over. NAME is your own label for this
# physical plate, so a run resumed from disk can be told apart from a fresh one.
NAME = "plate A"
routine = rt.Routine(dest, plan, name=NAME, strategy="by_row", path=PROGRESS)
routine.save()
print("new routine:", routine)

new routine: <Routine Destination.plate('corning_96_wellplate_360ul_flat' v2, slot 5, 96 wells) strategy='by_row' 0/1 objects>


### Routine — continue after a kernel restart

Run this after a restart. It loads the saved progress and prints a summary; read
it before confirming. Continuing is deliberate: the session refuses to start
until `confirm_resume()` is called, because software cannot tell a swapped fresh
plate from the original one. If you did change the plate, do **not** confirm —
start a new routine instead.

Two labware cases between runs: a **different format** (e.g. 96 → 384) is a
`move_labware(..., 'offDeck')` then a new `load_labware`, and the session's
labware check catches a mismatch or an empty slot on its own. The **same format,
a new plate** looks identical to continuing — that is what the confirmation
below guards.

In [11]:
routine = rt.Routine.load(PROGRESS)
print(routine.summary())

# Only after reading the summary above and checking the plate is the one this
# run started on. If you swapped in a fresh plate, do not confirm — make a new
# routine instead. Without this the session refuses to start.
routine.confirm_resume()

routine 'plate A' (run b0ad38f0, created 2026-08-11 00:44 UTC)
  0/1 objects delivered across 1 targets
  next: C3
  plate corning_96_wellplate_360ul_flat v2 in slot 5


In [13]:
# --- Logger (optional) --------------------------------------------------
logger = None      # run without logging

# To keep a run log, pass any object with a .log(str) method:
# import datetime
# class RunLog:
#     def __init__(self, path):
#         self.path = path
#     def log(self, msg):
#         with open(self.path, "a", encoding="utf-8") as f:
#             f.write(f"{datetime.datetime.now():%Y-%m-%d %H:%M:%S}  {msg}\n")
# logger = RunLog(paths.logs_dir() / f"picking_{PROFILE}.log")

### Run

`session.step()` runs in a worker thread while this cell's loop owns the window
and the keys. That split is deliberate: `pause` blocks *inside* `step()` between
moves, so a pause during a five-cuboid pickup takes effect at once. A single
thread that both stepped and read keys would deadlock the moment it paused, with
nothing left to read the un-pause key.

Keys (the window must have focus): **p** pause/resume, **Esc** stop the routine,
**q** leave the window without stopping it, **r** resume after `NEEDS_OPERATOR`.

In [57]:
# Teach where the crosshair disc is, once. Skip if it is already stored.
if "observe" not in profile.positions:
    jog("bring the crosshair disc under the camera, then Enter")
    profile.remember("observe", xyz(openapi))
print("observe:", profile.where("observe"))

observe: (281.87353703094635, 161.75594045486756, 114.50000000000001)


In [14]:
pause, stop = threading.Event(), threading.Event()

# Optional lower-camera clip per pickup. None = off: no recorder is created and
# no frames accumulate. To record, set a folder and make sure under_cam is open
# (section 4). A homography in the profile places the ROI box; without one the
# clip records with no box, which is not an error.
CLIP_DIR = paths.clips_dir() / "initial_test"                          # e.g. paths.clips_dir() / PROFILE

pmap = PixelMap.from_config(profile.pixel_map)
loaded = {slot: lid for slot, lid in openapi.labware_dct.items() if lid}
labware_id = loaded['5']
under_cam = cams.open("underview_cam", resolution = (2000,1500))
# off = profile.calibration.pipette_offset
# tip_offset = np.array([off.dx, off.dy])


session = PickingSession(openapi, over_cam, pmap, profile, routine,
                         cuboid_model, labware_id=labware_id, logger=logger,
                         under_cam=(under_cam if CLIP_DIR else None),
                         clip_dir=CLIP_DIR)

# The session writes its latest detections here whenever it emits a frame; the
# display reads them under the lock. The base video comes straight from the
# camera, so the window stays smooth between detections.
lock = threading.Lock()
snap = {"df": None, "pickable": None, "isolated": None, "zones": (),
        "state": session.state, "target": None}

def on_frame(frame, df):
    with lock:
        snap.update(df=df, pickable=session.pickable, isolated=session.isolated,
                    zones=session.floater_zones, state=session.state,
                    target=session.routine.current)
session.on_frame = on_frame

def worker():
    while not session.done:
        event = session.step(pause=pause, stop=stop)
        print(event)
        if session.state is RobotState.NEEDS_OPERATOR:
            time.sleep(0.2)                 # wait for the operator, do not spin

thread = threading.Thread(target=worker, daemon=True)
thread.start()

win = "picking"
cv2.namedWindow(win, cv2.WINDOW_NORMAL)
cv2.resizeWindow(win, 1348, 1011)
detach = False
try:
    while thread.is_alive():
        ok, frame = over_cam.read()
        if ok:
            with lock:
                s = dict(snap)
            lines = [f"state: {s['state'].value}",
                     f"target: {s['target']}",
                     "PAUSED" if pause.is_set() else "running"]
            vis = overlays.annotate(
                frame, cuboid_df=s["df"], pickable=s["pickable"],
                isolated=s["isolated"], floater_zones=s["zones"],
                floater_radius=cfg.floater_zone_radius_px,
                circle_center=cfg.circle_center, circle_radius=cfg.circle_radius,
                status_lines=lines)
            cv2.imshow(win, vis)

        key = cv2.waitKey(20) & 0xFF
        if key == ord("p"):
            pause.clear() if pause.is_set() else pause.set()
        elif key == 27:                     # Esc: stop the routine
            stop.set()
        elif key == ord("q"):               # leave the window, keep running
            detach = True
            break
        elif key == ord("r"):               # resume after NEEDS_OPERATOR
            session.resume()
finally:
    cv2.destroyWindow(win)
    if not detach:
        stop.set()
        thread.join(timeout=5)
        session.close()                     # detach the clip recorder, if any
        openapi.retract_axis("leftZ")
    print("state:", session.state.value, "(detached, still running)" if detach else "")

underview_cam: [0] Arducam B0478 (USB3 48MP)  2000x1500
  NOT applied: auto_exposure: asked 0.25, got 0, autofocus: asked 0, got 1, focus: asked 920, got 1
note: slot 5 reports corning_96_wellplate_360ul_flat v1, the routine recorded v2; ignoring, versions are definition revisions rather than plate identity
[capture_frame] idle: parked at the observation pose
[detect_floaters] captured: frame taken at the observation pose
[analyze_frame] floaters: skipped, within interval
[approach_target] analyzed: chose a batch {'isolated': 12, 'batch': 1}
[pickup_sample] approached: computed pickup coordinates {'n': 1}
[verify_pickup] picked: aspirated the batch {'n': 1}
[deposit_liquid_back] verified: checked the pickup {'attempted': 1, 'held': 0, 'missed': 1}
[transfer_to_well] deposited_back: returned volume to the dish {'volume': 10.0}
[capture_frame] transferred: deposited into the destination {'target': 'C3', 'volume': 0.0}
[detect_floaters] captured: frame taken at the observation pose
[analy

underview_cam: reopening (4000, 3000) -> (2000, 1500)
underview_cam: [0] Arducam B0478 (USB3 48MP)  2000x1500
  NOT applied: auto_exposure: asked 0.25, got 0, autofocus: asked 0, got 1, focus: asked 920, got 1


## 7. Shutting down

In [ ]:
openapi.retract_axis("leftZ")
cams.close_all()